# 17 — Interpretação, Robustez e Reprodutibilidade

> **Status:** notebook revisado e validado para estudo em JupyterLab.
>
> **Continuidade:** exemplos com dados sintéticos ou datasets integrados ao scikit-learn, sem acesso à internet.

**Objetivo geral:** construir compreensão conceitual e prática para aplicar este tema em problemas de Machine Learning, com avaliação adequada e interpretação dos resultados.


### Materiais complementares

- [Guia técnico — Interpretação, Robustez e Reprodutibilidade](./reference_materials/17_interpretabilidade_reprodutibilidade.pdf)
- [Cheat sheet — Interpretação, Robustez e Reprodutibilidade](./cheat_sheets/cheat_sheet_interpretabilidade_reprodutibilidade.pdf)
- [Documentação e resultados](./README.pt-BR.md)


## 1. Fundamentos, objetivos e contexto


Interpretação de modelo não é inferência causal. Importância por permutação mede queda de desempenho ao perturbar atributo sob métrica e dados definidos; atributos correlacionados podem dividir relevância. Avaliar por subgrupos pode revelar comportamento desigual, mas atributos sensíveis requerem cuidado ético e autorização. *Drift* pode tornar o desempenho histórico inadequado para dados futuros.


## 2. Fundamentos matemáticos

Importância por permutação de $j$: $I_j=\mathrm{score}(D,\hat f)-\frac1R\sum_{r=1}^R\mathrm{score}(\pi_{j,r}(D),\hat f)$. Estimativa condicionada ao conjunto avaliado, à métrica e à permutação, não um efeito causal.

**Conexão com disciplinas anteriores:** revise os conceitos matemáticos e estatísticos correspondentes. Não repetimos integralmente os capítulos de Python, visualização e inferência estatística.


## 3. Experimento guiado: preparando a pergunta

**Antes do código**, identifique: (a) quais dados são de treino, (b) qual informação poderá entrar no modelo, (c) que métrica ou propriedade será observada, e (d) que conclusão não é permitida pelos resultados. Estes exemplos são didáticos, com semente fixa onde aplicável.


In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score,confusion_matrix
import numpy as np
X,y=load_breast_cancer(return_X_y=True)
a,b,c,d=train_test_split(X,y,test_size=.3,stratify=y,random_state=42)
m=RandomForestClassifier(n_estimators=75,random_state=42,n_jobs=1).fit(a,c)
imp=permutation_importance(m,b,d,n_repeats=3,scoring='balanced_accuracy',random_state=42,n_jobs=1)
print('Top atributos por importância no teste (exploração após avaliação):',np.argsort(imp.importances_mean)[-5:][::-1])
print('Matriz de confusão:\n',confusion_matrix(d,m.predict(b)))

Top atributos por importância no teste (exploração após avaliação): [21  1 10 26 11]
Matriz de confusão:
 [[ 58   6]
 [  4 103]]


### Como interpretar

Os números são resultados específicos do experimento e da amostra, não garantias universais. Verifique o que mudou, qual método foi usado e qual é o cenário de aplicação. Compare com a teoria matemática e não confunda desempenho de treino, validação e teste.


## 4. Segundo experimento ou extensão controlada

Mude apenas uma condição principal (algoritmo, parâmetro ou tipo de dado), quando possível, e registre o efeito. Não ajuste decisões com base no teste final.


In [2]:
# Exemplo didático de análise por subgrupos artificiais (NÃO atributo sensível real)
subgrupo=(b[:,0]>np.median(a[:,0])).astype(int)
for g in [0,1]:
    idx=subgrupo==g
    print('Grupo artificial',g,'n',int(idx.sum()),'balanced_accuracy',round(balanced_accuracy_score(d[idx],m.predict(b[idx])),3))
# Registrar versões da execução sem gravar nada na máquina
import sklearn,pandas,numpy
print('Versões:',{'sklearn':sklearn.__version__,'pandas':pandas.__version__,'numpy':numpy.__version__})

Grupo artificial 0 n 81 balanced_accuracy 0.833
Grupo artificial 1 n 90 balanced_accuracy 0.89
Versões: {'sklearn': '1.9.1', 'pandas': '3.1.1', 'numpy': '2.5.4'}


## 5. Exercício resolvido

**Enunciado:** reforce o conceito central com um exemplo menor, cujo resultado pode ser inspecionado diretamente.


In [3]:
# Exercício resolvido: registrar informações mínimas de um experimento
experimento={'objetivo':'classificação binária','seed':42,'metrica':'balanced_accuracy','teste_reservado':True,'modelo':'RandomForestClassifier'}
for chave,valor in experimento.items(): print(chave,':',valor)

objetivo : classificação binária
seed : 42
metrica : balanced_accuracy
teste_reservado : True
modelo : RandomForestClassifier


## 6. Interpretação, limitações e erros comuns


**Perguntas de revisão:**

1. Qual hipótese implícita sustenta o procedimento usado?
2. Que erro de implementação ou de validação poderia tornar o resultado artificialmente otimista?
3. Como você mudaria o experimento para refletir dados reais?


## 7. Consulta rápida

- Interpretabilidade ≠ causalidade; descrever dataset, população e limitações.
- Fixar seeds, salvar métricas, registrar versões e origem dos dados.
- Model card básico: alvo, dados, métricas, subgrupos, limitações e uso indevido.
- **Não publicar** credenciais, dados restritos ou modelos binários não auditados.


## 8. Exercícios adicionais (para fazer no JupyterLab)

**Exercício A — sensibilidade:** altere um hiperparâmetro/hipótese e apresente uma tabela pequena com resultados e explicação. **Exercício B — revisão crítica:** construa um exemplo de uso incorreto (sem aplicá-lo a dados reais) e explique o procedimento correto. **Exercício C — comunicação:** escreva um parágrafo sobre escopo, métricas, suposições e limitações do método.

**Critério de aprovação:** códigos executam do começo ao fim, não há seleção pelo teste final e cada resultado possui interpretação.


## 9. Fontes e continuidade curricular


**Referências de atualização:** documentação de `scikit-learn` (<https://scikit-learn.org/stable/>); explicações matemáticas redigidas para esta reconstrução. Outros pacotes (Optuna, XGBoost, SHAP, imbalanced-learn) não são necessários para executar este capítulo.



## Aprofundamento v2 — auditoria e aplicação

**Ponto de atenção deste capítulo:** interpretação fora da amostra. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **interpretação fora da amostra** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — Permutation importance com conjunto independente

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.


In [4]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=300, n_features=5, n_informative=2, n_redundant=0, random_state=42)
Xt, Xv, yt, yv = train_test_split(X,y,test_size=.3,random_state=42,stratify=y)
m = RandomForestClassifier(n_estimators=40,random_state=42).fit(Xt,yt)
imp = permutation_importance(m,Xv,yv,n_repeats=5,random_state=42)
print('Importâncias médias:', np.round(imp.importances_mean, 3).tolist())

Importâncias médias: [0.004, 0.002, 0.047, 0.004, 0.431]


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **interpretação fora da amostra** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
